In [1]:
import os
from dotenv import load_dotenv
from typing import Literal, Optional, Any
from pydantic import BaseModel, Field
from utils.config_loader import load_config
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

class ConfigLoader:
    def __init__(self):
        self.config = load_config()

    def __getitem__(self, key):
        return self.config[key]

    
class ModelLoader(BaseModel):
    model_provider: Literal["groq", "openai"] = "groq"
    config: Optional[ConfigLoader] = Field(default=None, exclude=True)

    def model_post_init(self, __context: Any) -> None:
        self.config = ConfigLoader()
    
    class Config:
        arbitrary_types_allowed = True
    
    def load_llm(self):
        print("LLM loading...")

        groq_llm = ChatGroq(
            model=self.config["llm"]["groq"]["model_name"],
            api_key=os.getenv("GROQ_API_KEY")
        ).with_retry(stop_after_attempt=3)

        openai_llm = ChatOpenAI(
            model=self.config["llm"]["openai"]["model_name"],
            api_key=os.getenv("OPENAI_API_KEY")
        ).with_retry(stop_after_attempt=3)

        if self.model_provider == "groq":
            return groq_llm.with_fallbacks([openai_llm])

        elif self.model_provider == "openai":
            return openai_llm.with_fallbacks([groq_llm])

C:\Users\Dell\AppData\Local\Temp\ipykernel_35932\3251631478.py:17: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ModelLoader(BaseModel):


In [2]:
model_loader = ModelLoader(model_provider="groq")

FileNotFoundError: [Errno 2] No such file or directory: 'config/config.yaml'

In [3]:
import os

print(os.getcwd())

d:\real-time-projects\AI-Trip-Planner\notebook


In [4]:
print(os.listdir())

['experiments.ipynb', 'experiments2.ipynb', 'experiments3.ipynb']


In [6]:
print(os.getcwd())
print(os.path.exists("../config/config.yaml"))

d:\real-time-projects\AI-Trip-Planner\notebook
True


In [7]:
from pathlib import Path

project_root = Path.cwd().parent

config_path = project_root / "config" / "config.yaml"

print(config_path)
print(config_path.exists())

d:\real-time-projects\AI-Trip-Planner\config\config.yaml
True
